In [61]:
import os
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def analyze_concept_loss(base_dir, model, exp, method, clusters=['Cluster1', 'Cluster2', 'Cluster3'],filename=None):
    """
    Analyze which concepts are lost between different pruning percentages.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name (e.g., 'method1', 'method2')
        clusters: List of cluster names
    
    Returns:
        Dictionary containing loss analysis for each cluster
    """
    # Define pruning percentages
    pruning_percentages = ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
    
    results = {}
    
    for cluster in clusters:
        print(f"\n{'='*60}")
        print(f"Analyzing {cluster} for {model}/{exp}/{method}")
        print(f"{'='*60}")
        
        cluster_results = {
            'concepts_by_pruning': {},
            'loss_from_baseline': {},
            'sequential_loss': {},
            'cumulative_loss': {}
        }
        
        # Load concepts for each pruning percentage
        baseline_concepts = None
        prev_concepts = None
        
        for pruning_pct in pruning_percentages:
            filepath = os.path.join(base_dir, model, exp, method,filename, 'Expls', pruning_pct, f'{cluster}IOUS1024N.csv')
            
            if not os.path.exists(filepath):
                print(f"Warning: File not found - {filepath}")
                continue
            
            # Load unit-concept mappings
            unit_concepts = load_csv_data(filepath)
            
            # Get all unique concepts at this pruning level
            all_concepts = set()
            for concepts in unit_concepts.values():
                all_concepts.update(concepts)
            
            cluster_results['concepts_by_pruning'][pruning_pct] = {
                'all_concepts': all_concepts,
                'num_concepts': len(all_concepts),
                'unit_concepts': unit_concepts
            }
            
            # Set baseline (0.0% pruned)
            if baseline_concepts is None:
                baseline_concepts = all_concepts
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts (BASELINE)")
            else:
                # Calculate loss from baseline
                lost_from_baseline = baseline_concepts - all_concepts
                retained_from_baseline = baseline_concepts & all_concepts
                
                cluster_results['loss_from_baseline'][pruning_pct] = {
                    'lost_concepts': lost_from_baseline,
                    'num_lost': len(lost_from_baseline),
                    'retained_concepts': retained_from_baseline,
                    'num_retained': len(retained_from_baseline),
                    'loss_percentage': (len(lost_from_baseline) / len(baseline_concepts) * 100) if baseline_concepts else 0
                }
                
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts")
                print(f"  Lost from baseline: {len(lost_from_baseline)} ({cluster_results['loss_from_baseline'][pruning_pct]['loss_percentage']:.2f}%)")
                print(f"  Retained from baseline: {len(retained_from_baseline)}")
                
                # Calculate sequential loss (compared to previous pruning level)
                if prev_concepts is not None:
                    lost_sequential = prev_concepts - all_concepts
                    
                    cluster_results['sequential_loss'][pruning_pct] = {
                        'lost_concepts': lost_sequential,
                        'num_lost': len(lost_sequential),
                        'loss_percentage': (len(lost_sequential) / len(prev_concepts) * 100) if prev_concepts else 0
                    }
                    
                    print(f"  Lost since previous: {len(lost_sequential)} ({cluster_results['sequential_loss'][pruning_pct]['loss_percentage']:.2f}%)")
            
            prev_concepts = all_concepts
        
        results[cluster] = cluster_results
    
    return results
from collections import defaultdict
def generate_loss_summary(loss_results, save_path=None):
    """
    Generate a summary table of concept loss across pruning percentages.
    
    Args:
        loss_results: Results from analyze_concept_loss
        save_path: Optional path to save CSV summary
    """
    summary_data = []
    raw_concepts = defaultdict(lambda: defaultdict(list))
    max_s=defaultdict(int)
    for cluster, cluster_data in loss_results.items():
        for pct in cluster_data['loss_from_baseline'].keys():
            raw_concepts[cluster][pct]= list(cluster_data['loss_from_baseline'][pct]['lost_concepts'])
            max_s[cluster] = max(max_s[cluster], len(raw_concepts[cluster][pct]))
        for pruning_pct in cluster_data['concepts_by_pruning'].keys():
            row = {
                'Cluster': cluster,
                'Pruning_Percentage': pruning_pct,
                'Total_Concepts': cluster_data['concepts_by_pruning'][pruning_pct]['num_concepts']
            }
            
            if pruning_pct in cluster_data['loss_from_baseline']:
                row['Lost_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_lost']
                row['Lost_from_Baseline_Pct'] = cluster_data['loss_from_baseline'][pruning_pct]['loss_percentage']
                row['Retained_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_retained']
            else:
                row['Lost_from_Baseline'] = 0
                row['Lost_from_Baseline_Pct'] = 0
                row['Retained_from_Baseline'] = row['Total_Concepts']
            
            if pruning_pct in cluster_data['sequential_loss']:
                row['Lost_Sequential'] = cluster_data['sequential_loss'][pruning_pct]['num_lost']
                row['Lost_Sequential_Pct'] = cluster_data['sequential_loss'][pruning_pct]['loss_percentage']
            else:
                row['Lost_Sequential'] = 0
                row['Lost_Sequential_Pct'] = 0
            
            summary_data.append(row)
    
    for cluster in raw_concepts:
        for pct in raw_concepts[cluster]:
            dif = max_s[cluster] - len(raw_concepts[cluster][pct]) 
            if dif > 0:
                for i in range(dif):
                    raw_concepts[cluster][pct].append('')
                
        pd.DataFrame({k: sorted(v) for k, v in raw_concepts[cluster].items()}).to_csv(f"{save_path}_Cluster{cluster}_concepts_lost.csv")
    summary_df = pd.DataFrame(summary_data)
    pd.DataFrame(raw_concepts).to_csv("Raw_concepts_lost_to_pruning.csv")
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"\nSummary saved to {save_path}_concept_loss_summary.csv")
    
    return summary_df

def get_lost_concepts_details(loss_results, cluster, pruning_pct):
    """
    Get detailed list of lost concepts for a specific cluster and pruning percentage.
    
    Args:
        loss_results: Results from analyze_concept_loss
        cluster: Cluster name
        pruning_pct: Pruning percentage (e.g., '25.0%Pruned')
    
    Returns:
        Set of lost concepts
    """
    if cluster not in loss_results:
        print(f"Cluster {cluster} not found in results")
        return set()
    
    if pruning_pct not in loss_results[cluster]['loss_from_baseline']:
        print(f"Pruning percentage {pruning_pct} not found for {cluster}")
        return set()
    
    return loss_results[cluster]['loss_from_baseline'][pruning_pct]['lost_concepts']

# Example usage:
if __name__ == "__main__":
    # Set your paths
    base_dir = "/workspace/CCE_NLI"
    model = "LLAMA"
    exp = "exp"
    method = "lottery_ticket"
    filename='Run0.25_2'
    
    # Analyze concept loss
    loss_results = analyze_concept_loss(base_dir, model, exp, method, filename=filename)
    
    # Generate summary table
    summary_df = generate_loss_summary(loss_results, save_path=f'{model}_{method}_{filename}')
    print("\nSummary Table:")
    print(summary_df)

    # Get specific lost concepts
    lost_concepts_25 = get_lost_concepts_details(loss_results, 'Cluster1', '25.0%Pruned')
    print(f"\nConcepts lost at 25% pruning in cluster1: {len(lost_concepts_25)}")
    print(f"Examples: {list(lost_concepts_25)[:10]}")


Analyzing Cluster1 for LLAMA/exp/lottery_ticket

0.0%Pruned: 194 concepts (BASELINE)

25.0%Pruned: 135 concepts
  Lost from baseline: 80 (41.24%)
  Retained from baseline: 114
  Lost since previous: 80 (41.24%)

43.75%Pruned: 129 concepts
  Lost from baseline: 84 (43.30%)
  Retained from baseline: 110
  Lost since previous: 35 (25.93%)

57.812%Pruned: 158 concepts
  Lost from baseline: 77 (39.69%)
  Retained from baseline: 117
  Lost since previous: 28 (21.71%)

68.359%Pruned: 180 concepts
  Lost from baseline: 57 (29.38%)
  Retained from baseline: 137
  Lost since previous: 42 (26.58%)

76.27%Pruned: 170 concepts
  Lost from baseline: 65 (33.51%)
  Retained from baseline: 129
  Lost since previous: 56 (31.11%)

Analyzing Cluster2 for LLAMA/exp/lottery_ticket

0.0%Pruned: 233 concepts (BASELINE)

25.0%Pruned: 206 concepts
  Lost from baseline: 79 (33.91%)
  Retained from baseline: 154
  Lost since previous: 79 (33.91%)

43.75%Pruned: 221 concepts
  Lost from baseline: 76 (32.62%)
  Re

In [71]:
c1_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster1_concepts_lost.csv")
c1_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster1_concepts_lost.csv")
lost=set(c1_lost_lth['25.0%Pruned'])
for i in c1_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c1_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 32
Num lost in wanda and lth: 13
{'hyp:tok:green', 'pre:tok:top', 'pre:tok:there', 'pre:tok:lake', 'pre:tok:players', 'pre:tok:tree', 'hyp:tok:around', 'pre:tok:wall', 'pre:tok:performing', 'hyp:tok:driving', 'hyp:tok:female', 'hyp:tok:cats', 'pre:tok:out'}


In [67]:
len(preserved) #% of concepts that are lost once you rpune (lost at all iters)

33

In [72]:
c2_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster2_concepts_lost.csv")
c2_lost_lth= pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster2_concepts_lost.csv")
lost=set(c2_lost_lth['25.0%Pruned'])
for i in c2_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c2_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 21
Num lost in wanda and lth: 8
{'pre:tok:church', 'pre:tok:vendor', 'pre:tok:workers', 'hyp:tok:air', 'hyp:tok:guitar', 'pre:tok:jacket', 'hyp:tok:couple', 'pre:tag:ex'}


In [73]:
c3_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster3_concepts_lost.csv")
c3_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster3_concepts_lost.csv")
lost=set(c3_lost_lth['25.0%Pruned'])
for i in c3_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c3_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 27
Num lost in wanda and lth: 10
{'hyp:tok:construction', 'pre:tok:graffiti', 'hyp:tok:summer', 'pre:tok:skier', 'pre:tok:board', 'pre:tok:surfing', 'pre:tok:shoulders', 'pre:tok:martial', 'pre:tok:flying', 'hyp:tok:runs'}


In [123]:
import os
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def get_all_concepts_across_clusters(base_dir, model, exp, method, pruning_pct, 
                                     clusters=['cluster1', 'cluster2', 'cluster3'],filename=None):
    """
    Get all unique concepts across all clusters for a given pruning percentage.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name
        pruning_pct: Pruning percentage (e.g., '0.0%Pruned')
        clusters: List of cluster names
    
    Returns:
        Set of all unique concepts across all clusters
    """
    all_concepts = set()
    
    for cluster in clusters:
        filepath = os.path.join(base_dir, model, exp, method,filename,   'Expls',pruning_pct, f'{cluster}IOUS1024N.csv')
        
        if not os.path.exists(filepath):
            print(f"Warning: File not found - {filepath}")
            continue
        
        # Load unit-concept mappings
        unit_concepts = load_csv_data(filepath)
        
        # Get all concepts in this cluster
        for concepts in unit_concepts.values():
            all_concepts.update(concepts)
    
    return all_concepts

def analyze_concept_loss_across_clusters(base_dir, model, exp, method, clusters=['Cluster1', 'Cluster2', 'Cluster3'],filename=None):
    """
    Analyze which concepts are lost across ALL clusters between different pruning percentages.
    A concept is "lost" if it appears in ANY cluster at 0.0%Pruned but doesn't appear 
    in ANY cluster at higher pruning percentages.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name
        clusters: List of cluster names
    
    Returns:
        Dictionary containing loss analysis across all clusters
    """

    # Define pruning percentages
    pruning_percentages = ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
    
    print(f"\n{'='*60}")
    print(f"Analyzing concept loss ACROSS ALL CLUSTERS for {model}/{exp}/{method}")
    print(f"{'='*60}")
    
    results = {
        'concepts_by_pruning': {},
        'loss_from_baseline': {},
        'sequential_loss': {},
        'foundational':set()
    }
    
    # Load concepts for each pruning percentage (across all clusters)
    baseline_concepts = None
    prev_concepts = None
    
    for pruning_pct in pruning_percentages:
        # Get all concepts across all clusters for this pruning percentage
        all_concepts = get_all_concepts_across_clusters(base_dir, model, exp, method, 
                                                        pruning_pct, clusters,filename)
        
        results['concepts_by_pruning'][pruning_pct] = {
            'all_concepts': all_concepts,
            'num_concepts': len(all_concepts)
        }
        
        # Set baseline (0.0% pruned)
        if baseline_concepts is None:
            baseline_concepts = all_concepts
            print(f"\n{pruning_pct}: {len(all_concepts)} concepts (BASELINE)")
            print(f"  Concepts appear across any of the clusters: {clusters}")
            results['foundational'] = baseline_concepts
        else:
            # Calculate loss from baseline
            lost_from_baseline = baseline_concepts - all_concepts
            retained_from_baseline = baseline_concepts & all_concepts
            results['foundational'] = results['foundational'].intersection(all_concepts)
            
            results['loss_from_baseline'][pruning_pct] = {
                'lost_concepts': lost_from_baseline,
                'num_lost': len(lost_from_baseline),
                'retained_concepts': retained_from_baseline,
                'num_retained': len(retained_from_baseline),
                'loss_percentage': (len(lost_from_baseline) / len(baseline_concepts) * 100) if baseline_concepts else 0,
                'num_foundational': len(lottery_ticket_foundational[0].intersection(all_concepts))/len(lottery_ticket_foundational[0]),
            }
            
            print(f"\n{pruning_pct}: {len(all_concepts)} concepts")
            print(f"  Lost from baseline (not in ANY cluster): {len(lost_from_baseline)} ({results['loss_from_baseline'][pruning_pct]['loss_percentage']:.2f}%)")
            print(f"  Retained from baseline (in at least one cluster): {len(retained_from_baseline)}")
            
            # Calculate sequential loss (compared to previous pruning level)
            if prev_concepts is not None:
                lost_sequential = prev_concepts - all_concepts
                
                results['sequential_loss'][pruning_pct] = {
                    'lost_concepts': lost_sequential,
                    'num_lost': len(lost_sequential),
                    'loss_percentage': (len(lost_sequential) / len(prev_concepts) * 100) if prev_concepts else 0
                }
                
                print(f"  Lost since previous pruning level: {len(lost_sequential)} ({results['sequential_loss'][pruning_pct]['loss_percentage']:.2f}%)")
        
        prev_concepts = all_concepts
    
    return results

def generate_loss_summary(loss_results, save_path=None):
    """
    Generate a summary table of concept loss across pruning percentages.
    
    Args:
        loss_results: Results from analyze_concept_loss_across_clusters
        save_path: Optional path to save CSV summary
    """
    summary_data = []
    print(len(loss_results['foundational']))
    for pruning_pct in loss_results['concepts_by_pruning'].keys():
        row = {
            'Pruning_Percentage': pruning_pct,
            'Total_Concepts': loss_results['concepts_by_pruning'][pruning_pct]['num_concepts']
        }
        
        if pruning_pct in loss_results['loss_from_baseline']:
            row['Lost_from_Baseline'] = loss_results['loss_from_baseline'][pruning_pct]['num_lost']
            row['Lost_from_Baseline_Pct'] = loss_results['loss_from_baseline'][pruning_pct]['loss_percentage']
            row['Retained_from_Baseline'] = loss_results['loss_from_baseline'][pruning_pct]['num_retained']
            row['Foundational'] = loss_results['loss_from_baseline'][pruning_pct]['num_foundational']
        else:
            row['Lost_from_Baseline'] = 0
            row['Lost_from_Baseline_Pct'] = 0
            row['Retained_from_Baseline'] = row['Total_Concepts']
        
        if pruning_pct in loss_results['sequential_loss']:
            row['Lost_Sequential'] = loss_results['sequential_loss'][pruning_pct]['num_lost']
            row['Lost_Sequential_Pct'] = loss_results['sequential_loss'][pruning_pct]['loss_percentage']
        else:
            row['Lost_Sequential'] = 0
            row['Lost_Sequential_Pct'] = 0
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"\nSummary saved to {save_path}")
    
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    print(summary_df.to_string(index=False))
    
    return summary_df

def save_lost_concepts_details(loss_results, save_dir='lost_concepts_details'):
    """
    Save detailed lists of lost concepts to files.
    
    Args:
        loss_results: Results from analyze_concept_loss_across_clusters
        save_dir: Directory to save the detailed files
    """
    os.makedirs(save_dir, exist_ok=True)
    lost={}
    max_len=0
    for pruning_pct in loss_results['loss_from_baseline'].keys():
        lost_concepts = loss_results['loss_from_baseline'][pruning_pct]['lost_concepts']
        lost[pruning_pct] = sorted(list(lost_concepts))
        max_len = max(max_len, len(lost[pruning_pct]))
    
    for pct in lost:
        dif = max_len-len(lost[pct]) 
        if dif > 0:
            for _ in range(dif):
                lost[pct].append('')
    pd.DataFrame(lost).to_csv(f"lost_cps.csv")
        
       
        
        #print(f"Saved lost concepts to {filename}")

# Example usage:

# Set your paths

base_dir = "/workspace/CCE_NLI"
model = "LLAMA"
exp = "exp"
method = "wanda"
filename='Run0.25_2'

# Analyze concept loss across all clusters
loss_results = analyze_concept_loss_across_clusters(base_dir, model, exp, method,filename=filename)

# Generate summary table
summary_df = generate_loss_summary(loss_results, save_path='concept_loss_summary.csv')

# Save detailed lists of lost concepts
save_lost_concepts_details(loss_results, save_dir='lost_concepts_details')

# Access specific information
print("\n" + "="*80)
print("EXAMPLE: Concepts lost at 25.0%Pruned")
print("="*80)
lost_at_25 = loss_results['loss_from_baseline']['25.0%Pruned']['lost_concepts']
print(f"Total lost: {len(lost_at_25)}")
print(f"First 10 examples: {list(sorted(lost_at_25))[:10]}")


Analyzing concept loss ACROSS ALL CLUSTERS for LLAMA/exp/wanda

0.0%Pruned: 431 concepts (BASELINE)
  Concepts appear across any of the clusters: ['Cluster1', 'Cluster2', 'Cluster3']

25.0%Pruned: 435 concepts
  Lost from baseline (not in ANY cluster): 67 (15.55%)
  Retained from baseline (in at least one cluster): 364
  Lost since previous pruning level: 67 (15.55%)

43.75%Pruned: 410 concepts
  Lost from baseline (not in ANY cluster): 110 (25.52%)
  Retained from baseline (in at least one cluster): 321
  Lost since previous pruning level: 102 (23.45%)

57.812%Pruned: 335 concepts
  Lost from baseline (not in ANY cluster): 152 (35.27%)
  Retained from baseline (in at least one cluster): 279
  Lost since previous pruning level: 132 (32.20%)

68.359%Pruned: 258 concepts
  Lost from baseline (not in ANY cluster): 200 (46.40%)
  Retained from baseline (in at least one cluster): 231
  Lost since previous pruning level: 114 (34.03%)

76.27%Pruned: 191 concepts
  Lost from baseline (not in 

In [102]:
import pandas as pd
lost_concepts= pd.read_csv("/workspace/CCE_NLI/Experiments/lost_concepts_details/lost_cps_llama_lth.csv")
all_lost=set()
all_cps=set()
for i,col in enumerate(lost_concepts.columns):
    if 'Unnamed' in col: continue
    cps = get_concepts(lost_concepts[col])
    if i == 1:
        all_lost=cps
        all_cps=cps
    else:
        all_lost = all_lost.intersection(cps)
        all_cps=all_cps.union(cps)
len(all_lost), len(all_cps)

(33, 173)

In [ ]:
LLAMA wanda Run0.25_2        % of Foundational concpts expld (foundational meaning concepts that were preserved ebd to end in lth pruning)
        0.0%Pruned            NaN
       25.0%Pruned            0.980695
      43.75%Pruned            0.953668
     57.812%Pruned            0.884170
     68.359%Pruned            0.783784
      76.27%Pruned            0.633205

In [95]:
def get_concepts(col):
    cps=set()
    for row in col:
        cps.add(row)
    return cps

In [119]:
lottery_ticket_foundational[0]

{'hyp:tag:.',
 'hyp:tag:cc',
 'hyp:tag:cd',
 'hyp:tag:dt',
 'hyp:tag:ex',
 'hyp:tag:in',
 'hyp:tag:jj',
 'hyp:tag:nn',
 'hyp:tag:nnp',
 'hyp:tag:nns',
 'hyp:tag:prp',
 'hyp:tag:prp$',
 'hyp:tag:rb',
 'hyp:tag:vb',
 'hyp:tag:vbd',
 'hyp:tag:vbg',
 'hyp:tag:vbn',
 'hyp:tag:vbp',
 'hyp:tag:vbz',
 'hyp:tok:after',
 'hyp:tok:alone',
 'hyp:tok:and',
 'hyp:tok:are',
 'hyp:tok:asleep',
 'hyp:tok:at',
 'hyp:tok:baseball',
 'hyp:tok:beach',
 'hyp:tok:because',
 'hyp:tok:bed',
 'hyp:tok:bike',
 'hyp:tok:black',
 'hyp:tok:blue',
 'hyp:tok:boy',
 'hyp:tok:bus',
 'hyp:tok:car',
 'hyp:tok:cat',
 'hyp:tok:chasing',
 'hyp:tok:child',
 'hyp:tok:children',
 'hyp:tok:competition',
 'hyp:tok:cooking',
 'hyp:tok:dancing',
 'hyp:tok:dog',
 'hyp:tok:dogs',
 'hyp:tok:driving',
 'hyp:tok:eating',
 'hyp:tok:enjoying',
 'hyp:tok:first',
 'hyp:tok:for',
 'hyp:tok:friends',
 'hyp:tok:game',
 'hyp:tok:girl',
 'hyp:tok:girls',
 'hyp:tok:guy',
 'hyp:tok:has',
 'hyp:tok:her',
 'hyp:tok:his',
 'hyp:tok:home',
 'hyp:tok: